***What is an N-Gram*** -
An N gram is a sequence of N items (Usually words or characters) taken from a text. The "N" is just a number - it teels you how many items to group together at a time.

UNIGRAM (N = 1) : single words - ["I" , "Love" , "NLP"]
BIGRAM (N = 2) : two word pairs - ["I LOVE" , "LOVE NLP"]
TRIGRAM (N = 3) : three word sequences - ["I LOVE NLP"]

***WHY N GRAMS ARE USED?*** - Words alone (Unigrams) loose context. consider these two sentences : "Not Good" vs "Good". A unigram model sees only "Good" and misses the negative entirely. N grams solves this by bundling neighboring words.

***Practical implementation of N-grams***

In [ ]:
import nltk
from nltk.util import ngrams
from nltk.tokenize import word_tokenize

In [ ]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
text = " The movie was not good at all but the acting was brilliant"
tokens = word_tokenize(text.lower())

In [ ]:
#Generate bigrams and trigrams
bigrams_list = list(ngrams(tokens,2))
trigrams_list = list(ngrams(tokens,3))

In [ ]:
print("Tokens:", tokens)
print("\nBigrams:")
for bg in bigrams_list:
  print(" ",bg)

Tokens: ['the', 'movie', 'was', 'not', 'good', 'at', 'all', 'but', 'the', 'acting', 'was', 'brilliant']

Bigrams:
  ('the', 'movie')
  ('movie', 'was')
  ('was', 'not')
  ('not', 'good')
  ('good', 'at')
  ('at', 'all')
  ('all', 'but')
  ('but', 'the')
  ('the', 'acting')
  ('acting', 'was')
  ('was', 'brilliant')


In [ ]:
print("\nTrigrams:")
for tg in trigrams_list:
    print(" ", tg)


Trigrams:
  ('the', 'movie', 'was')
  ('movie', 'was', 'not')
  ('was', 'not', 'good')
  ('not', 'good', 'at')
  ('good', 'at', 'all')
  ('at', 'all', 'but')
  ('all', 'but', 'the')
  ('but', 'the', 'acting')
  ('the', 'acting', 'was')
  ('acting', 'was', 'brilliant')


***Counting and filtering N-grams***

Raw N Grams are only useful when you count their frequencies and filter out noise. This is where FreqDist and counter Come in.

In [ ]:
import nltk
from collections import Counter
from nltk.util import ngrams
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
text = """
The product is amazing. I love the product. The product quality is really good.
The product is worth buying. I would recommend the product to everyone.
Not a good product. The product failed me badly.
"""

In [ ]:
#Preproccessing
tokens = word_tokenize(text.lower())
stop_words = set(stopwords.words('english'))

In [ ]:
# Function to get top bigrams
def get_bigrams(tokens,filter_sttopwords=False):
  processed_tokens = tokens
  if filter_sttopwords:
    processed_tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
  return Counter(ngrams(processed_tokens , 2))

In [ ]:
# Raw bigrams
raw_bigrams = get_bigrams(tokens)
print("Top 5 raw bigrams:")
for k , v in raw_bigrams.most_common(5):
  print(k,":",v)


Top 5 raw bigrams:
('the', 'product') : 6
('.', 'the') : 3
('product', 'is') : 2
('.', 'i') : 2
('product', '.') : 2


In [ ]:
#Filtered bigrams
filtered_bigrams = get_bigrams(tokens,True)
print("\nTop 5 filtered bigrams:")
for k , v in filtered_bigrams.most_common(5):
  print(k , ":" , v)


Top 5 filtered bigrams:
('product', 'product') : 2
('good', 'product') : 2
('product', 'amazing') : 1
('amazing', 'love') : 1
('love', 'product') : 1


In [ ]:
#Bigrams with min frequency
min_freq = 2
common = {k: v for k ,v in filtered_bigrams.items() if v >= min_freq}
print(f"\nBigrams appearing >= {min_freq} times:")
for k , v in common.items():
  print( k , ":", v)


Bigrams appearing >= 2 times:
('product', 'product') : 2
('good', 'product') : 2


***Analysing Bigrams to provide context in Sentiment Analysis***

In [ ]:
from nltk.util import ngrams
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
# --- Sentiment lexicons ---
positive_words = {
    'good', 'great', 'amazing', 'excellent', 'love',
    'wonderful', 'fantastic', 'best', 'brilliant', 'happy'
}

negative_words = {
    'bad', 'terrible', 'awful', 'horrible', 'hate',
    'worst', 'poor', 'disappointing', 'boring', 'sad'
}

In [ ]:
# --- Bigram context rules ---
# These flip or amplify the base sentiment
negation_bigrams = {
    'not good', 'not great', 'not bad', 'not amazing',
    'never good', 'not happy', 'not excellent'
}

intensifier_bigrams = {
    'very good', 'really great', 'extremely bad',
    'absolutely terrible', 'so amazing', 'quite poor'
}

In [ ]:
def analyze_sentiment_unigram(text):
    tokens = word_tokenize(text.lower())
    score = 0
    for token in tokens:
        if token in positive_words:
            score += 1
        elif token in negative_words:
            score -= 1
    return score


def analyze_sentiment_bigram(text):
    tokens = word_tokenize(text.lower())
    score = 0
    bigram_list = list(ngrams(tokens, 2))
    bigram_strings = {' '.join(bg) for bg in bigram_list}

    handled_indices = set()  # track tokens already handled by a bigram

    for i, (w1, w2) in enumerate(bigram_list):
        pair = f"{w1} {w2}"
        if pair in negation_bigrams:
            # Flip the expected sentiment
            if w2 in positive_words:
                score -= 1   # "not good" -> negative despite 'good'
            elif w2 in negative_words:
                score += 1   # "not bad" -> positive despite 'bad'
            handled_indices.update([i, i+1])
        elif pair in intensifier_bigrams:
            # Amplify sentiment
            if w2 in positive_words:
                score += 2
            elif w2 in negative_words:
                score -= 2
            handled_indices.update([i, i+1])

    # Handle remaining tokens not covered by bigrams
    for i, token in enumerate(tokens):
        if i not in handled_indices:
            if token in positive_words:
                score += 1
            elif token in negative_words:
                score -= 1

    return score


In [ ]:
# --- Test sentences ---
test_sentences = [
    "The movie was good",
    "The movie was not good",
    "The movie was not bad",
    "The food was very good",
    "The service was absolutely terrible",
    "I love this product but the shipping was horrible",
]

print(f"{'Sentence':<50} {'Unigram':>8} {'Bigram':>8} {'Difference':>12}")
print("-" * 82)
for sentence in test_sentences:
    u = analyze_sentiment_unigram(sentence)
    b = analyze_sentiment_bigram(sentence)
    diff = "DIFFERS" if u != b else "same"
    print(f"{sentence:<50} {u:>8} {b:>8} {diff:>12}")

Sentence                                            Unigram   Bigram   Difference
----------------------------------------------------------------------------------
The movie was good                                        1        1         same
The movie was not good                                    1       -1      DIFFERS
The movie was not bad                                    -1        1      DIFFERS
The food was very good                                    1        2      DIFFERS
The service was absolutely terrible                      -1       -2      DIFFERS
I love this product but the shipping was horrible         0        0         same


***Advance deployment***

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np

In [ ]:
# Training data
reviews = [
    ("The food was amazing and delicious", 1),
    ("I absolutely love this restaurant", 1),
    ("Best meal I have ever had", 1),
    ("Really good service and great ambiance", 1),
    ("The staff was wonderful and friendly", 1),
    ("Excellent quality, highly recommend", 1),
    ("The food was terrible and cold", 0),
    ("I hate this place, never coming back", 0),
    ("Worst experience ever, disgusting food", 0),
    ("Not good at all, very disappointing", 0),
    ("The service was horrible and rude", 0),
    ("Terrible food and not worth the price", 0),
    ("Not bad but not great either", 1),    # Tricky: mildly positive
    ("Could be better but overall okay", 1),
    ("Extremely disappointing, not recommended", 0)
]

In [ ]:
texts, labels = zip(*reviews)
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.3, random_state=42)

In [ ]:
# --- Model 1: Unigram only ---
unigram_model = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(1, 1))),
    ('classifier', MultinomialNB())
])

In [ ]:
# --- Model 2: Bigram + Unigram ---
bigram_model = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(1, 2))),  # includes both 1-grams and 2-grams
    ('classifier', MultinomialNB())
])

In [ ]:
# --- Model 3: TF-IDF with bigrams (even better) ---
tfidf_bigram_model = Pipeline([
    ('vectorizer', TfidfVectorizer(ngram_range=(1, 2))),
    ('classifier', MultinomialNB())
])

models = {
    'Unigram':        unigram_model,
    'Bigram':         bigram_model,
    'TF-IDF Bigram':  tfidf_bigram_model,
}

In [ ]:
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    accuracy = np.mean(np.array(preds) == np.array(y_test))
    print(f"\n{name} model accuracy: {accuracy:.2%}")
    print(classification_report(y_test, preds, target_names=['Negative', 'Positive']))


Unigram model accuracy: 40.00%
              precision    recall  f1-score   support

    Negative       0.33      0.50      0.40         2
    Positive       0.50      0.33      0.40         3

    accuracy                           0.40         5
   macro avg       0.42      0.42      0.40         5
weighted avg       0.43      0.40      0.40         5


Bigram model accuracy: 40.00%
              precision    recall  f1-score   support

    Negative       0.33      0.50      0.40         2
    Positive       0.50      0.33      0.40         3

    accuracy                           0.40         5
   macro avg       0.42      0.42      0.40         5
weighted avg       0.43      0.40      0.40         5


TF-IDF Bigram model accuracy: 40.00%
              precision    recall  f1-score   support

    Negative       0.33      0.50      0.40         2
    Positive       0.50      0.33      0.40         3

    accuracy                           0.40         5
   macro avg       0.42    

In [ ]:
vectorizer = CountVectorizer(ngram_range=(1, 2))
vectorizer.fit(texts)

CountVectorizer(ngram_range=(1, 2))

In [ ]:
# Show all bigram features
all_features = vectorizer.get_feature_names_out()
bigram_features = [f for f in all_features if ' ' in f]
print("Learned bigram features:")
for f in bigram_features[:20]:
    print(" ", f)

Learned bigram features:
  absolutely love
  all very
  amazing and
  and cold
  and delicious
  and friendly
  and great
  and not
  and rude
  at all
  bad but
  be better
  best meal
  better but
  but not
  but overall
  coming back
  could be
  disappointing not
  disgusting food


Write a function ngram_freq(text, n) that takes a string and an integer N, tokenizes the text, generates all N-grams, and returns a dictionary mapping each N-gram tuple to its frequency. Test it on the sample text with N=2.

In [ ]:
import nltk
from nltk.util import ngrams
from nltk.tokenize import word_tokenize
from collections import Counter

In [ ]:
nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
def ngram_freq(text , n):
  tokens = word_tokenize(text.lower())
  n_grams = ngrams(tokens,n)
  return dict(Counter(n_grams))

In [ ]:
text = """
The product is amazing. I love the product. The product quality is really good.
The product is worth buying. I would recommend the product to everyone.
Not a good product. The product failed me badly.
"""

In [ ]:
result = ngram_freq(text,2)

In [ ]:
#Print top 5 bigrams
print("Top 5 bigrams:")
for gram , count in sorted(result.items(),key = lambda x: -x[1])[:5]:
  print(gram,":",count)

Top 5 bigrams:
('the', 'product') : 6
('.', 'the') : 3
('product', 'is') : 2
('.', 'i') : 2
('product', '.') : 2


***Sentiment analysis with negation bigrams*** -  Build a sentiment scorer that detects negation bigrams and flips sentiment accordingly. Compare it against a naive unigram model.

In [ ]:
import re
from collections import Counter

In [ ]:
POSITIVE = {'good', 'great', 'excellent', 'amazing', 'love', 'brilliant',
            'fantastic', 'wonderful', 'best', 'happy', 'enjoyed', 'perfect'}

NEGATIVE = {'bad', 'terrible', 'awful', 'horrible', 'hate', 'worst',
            'poor', 'disappointing', 'boring', 'dull', 'sad', 'failed'}

NEGATORS  = {'not', 'never', 'no', 'neither', 'hardly', 'barely'}
INTENSIFIERS = {'very', 'really', 'extremely', 'absolutely', 'so', 'quite'}

In [ ]:
def tokenize(text):
  return re.sub(r"[^\w\s]","",text.lower()).split()

def unigram_score(text):
  score = 0
  for word in tokenize(text):
    if word in POSITIVE: score += 1
    elif word in NEGATIVE: score -= 1
  return score

def bigram_score(text):
  tokens = tokenize(text)
  score = 0
  skip_next = False

  for i , word in enumerate(tokens):
    if skip_next:
      skip_next = False
      continue

    next_word = tokens[i+1] if i + 1 < len(tokens) else None

    # Check for negation bigram
    if word in NEGATORS and next_word:
      if next_word in POSITIVE:
        score -= 1
      elif next_word in NEGATIVE:
        score += 1
      skip_next = True

    # Check for intensifiers
    elif word in INTENSIFIERS and next_word:
      if next_word in POSITIVE:
        score += 2
      elif next_word in NEGATIVE:
        score -= 2
      skip_next = True

    # Plain Unigram fallback for the current word if not handled by a bigram
    else:
      if word in POSITIVE: score += 1
      elif word in NEGATIVE: score -= 1
  return score

In [ ]:
def label(score):
  if score > 0 : return f"POSITIVE (+{score})"
  if score > 0 : return f"NEGATIVE (+{score})"
  return "NEUTRAL(0)"

In [ ]:
sentences = [
    "The movie was good",
    "The movie was not good",        # negation flips
    "The food was not bad",          # double negative → positive
    "The service was very good",     # intensifier
    "It was absolutely terrible",    # intensifier + negative
    "Not great but not horrible either",
    "I never hate a good story",
    "The acting was brilliant",
    "Really bad and extremely poor quality",
    "Hardly good and barely enjoyable",
]

In [ ]:
print(f"{'Sentence':<45} {'Unigram':>12} {'Bigram':>12}  Match?")
print("─" * 78)
for s in sentences:
    u = unigram_score(s)
    b = bigram_score(s)
    match = "✓" if u == b else "← differs"
    print(f"{s:<45} {label(u):>12} {label(b):>12}  {match}")

Sentence                                           Unigram       Bigram  Match?
──────────────────────────────────────────────────────────────────────────────
The movie was good                            POSITIVE (+1) POSITIVE (+1)  ✓
The movie was not good                        POSITIVE (+1)   NEUTRAL(0)  ← differs
The food was not bad                            NEUTRAL(0) POSITIVE (+1)  ← differs
The service was very good                     POSITIVE (+1) POSITIVE (+2)  ← differs
It was absolutely terrible                      NEUTRAL(0)   NEUTRAL(0)  ← differs
Not great but not horrible either               NEUTRAL(0)   NEUTRAL(0)  ✓
I never hate a good story                       NEUTRAL(0) POSITIVE (+2)  ← differs
The acting was brilliant                      POSITIVE (+1) POSITIVE (+1)  ✓
Really bad and extremely poor quality           NEUTRAL(0)   NEUTRAL(0)  ← differs
Hardly good and barely enjoyable              POSITIVE (+1)   NEUTRAL(0)  ← differs
